# Golden Dataset Evaluation
Compares trained model checkpoints on a held-out golden dataset — images sourced independently from the training data to give an honest, out-of-distribution accuracy estimate.

Each model was trained under different dataset / augmentation conditions; the golden set is fixed across all comparisons so the results are directly comparable.

| Model label | Dataset | Augmentation | Notes |
|---|---|---|---|
| Dirty | Full raw dataset (~29k images) | Yes | Includes near-duplicates and static pre-augmented images |
| in question | No exact duplicates (~25.8k) | Yes | 3,381 duplicates removed |
| No dups+aug | No duplicates | Yes | Random GPU augmentation |
| No dups | No duplicates | No | Baseline clean dataset |

In [1]:
import subprocess
subprocess.run(["pip", "install", "nbformat", "--upgrade"], check=True)


CompletedProcess(args=['pip', 'install', 'nbformat', '--upgrade'], returncode=0)

## 1. Setup
Load each model checkpoint and run inference on the golden dataset. All models share the same EfficientNet_V2_S architecture and IMAGENET1K_V1 pretrained weights — only the training data and augmentation strategy differ.

In [2]:
# ── Multi-model golden dataset comparison ──────────────────────────────────────
import torch
import torch.nn as nn
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torchvision.models import get_model, get_weight
from torch.utils.data import DataLoader
from utils.dataset import ProduceDataset
from pathlib import Path
from tqdm import tqdm

GOLDEN_PATH =  Path(".") / ".."/ "golden_dataset" 
NUM_CLASSES = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS = {
    "Dirty":          "models/EX4_efficientnet_finetune_adam_cw_20260417190029_dirty.pth",
    "No identical":   "models/EX4_efficientnet_finetune_adam_cw_20260418140103_no_identical.pth",
    "No aug":         "models/EX5_efficientnet_finetune_adam_cw_noaug_20260419170823_no_aug.pth",
    "Progressive":    "models/EX4_efficientnet_finetune_adam_cw_20260419145751_progressive_aug.pth",
    "No dups":        "models/EX4_efficientnet_finetune_adam_cw_20260417194945_no_dups.pth",
}

def load_model(path):
    pretrained_weights = get_weight("EfficientNet_V2_S_Weights.IMAGENET1K_V1")
    m = get_model("efficientnet_v2_s", weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    checkpoint = torch.load(path, weights_only=False)
    m.load_state_dict(checkpoint["model_state_dict"])
    m.to(device).eval()
    return m, pretrained_weights.transforms()

def evaluate(model, transforms):
    dataset = ProduceDataset(root_dir=GOLDEN_PATH, transform=transforms)
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.max(dim=1).values.cpu().tolist())
    all_paths = dataset.image_paths
    return pd.DataFrame({
        "produce": [Path(p).parent.name.split("__")[0] for p in all_paths],
        "true_label": all_labels,
        "pred": all_preds,
        "correct": [p == l for p, l in zip(all_preds, all_labels)],
        "confidence": all_probs,
    })


# ── Run evaluation ─────────────────────────────────────────────────────────────
results = {}
for name, path in MODELS.items():
    print(f"Evaluating: {name}")
    m, transforms = load_model(path)
    results[name] = evaluate(m, transforms)
    print(f"  Overall: {results[name]['correct'].mean():.4f}")

# ── Overall accuracy bar chart ─────────────────────────────────────────────────
overall = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [df["correct"].mean() for df in results.values()]
})

fig1 = px.bar(overall, x="Model", y="Accuracy", text_auto=".3f",
              title="Overall accuracy — golden dataset",
              color="Model", range_y=[0.5, 1.0])
fig1.update_traces(textposition="outside")
fig1.show()

# ── Per-category grouped bar chart ────────────────────────────────────────────
per_cat = []
for name, df in results.items():
    cat_acc = df.groupby("produce")["correct"].mean().reset_index()
    cat_acc.columns = ["produce", "accuracy"]
    cat_acc["model"] = name
    per_cat.append(cat_acc)

per_cat_df = pd.concat(per_cat)

fig2 = px.bar(per_cat_df, x="produce", y="accuracy", color="model",
              barmode="group", title="Per-category accuracy — golden dataset",
              range_y=[0, 1.0], text_auto=".2f")
fig2.update_layout(xaxis_tickangle=-45, height=500)
fig2.show()

# ── Heatmap ───────────────────────────────────────────────────────────────────
pivot = per_cat_df.pivot(index="model", columns="produce", values="accuracy")

fig3 = px.imshow(pivot, text_auto=".2f", aspect="auto",
                 color_continuous_scale="RdYlGn", range_color=[0.5, 1.0],
                 title="Accuracy heatmap — model vs category")
fig3.show()


Evaluating: Dirty
  Overall: 0.8396
Evaluating: No identical
  Overall: 0.7987
Evaluating: No aug
  Overall: 0.9119
Evaluating: Progressive
  Overall: 0.7704
Evaluating: No dups
  Overall: 0.7327


In [3]:
from sklearn.metrics import precision_recall_fscore_support

# ── Overall F1 bar chart ───────────────────────────────────────────────────────
overall_f1 = []
for name, df in results.items():
    _, _, f1, _ = precision_recall_fscore_support(
        df["true_label"], df["pred"], average="macro", zero_division=0
    )
    overall_f1.append({"Model": name, "F1": f1})

overall_f1_df = pd.DataFrame(overall_f1)

fig_f1_overall = px.bar(overall_f1_df, x="Model", y="F1", text_auto=".3f",
                        title="Overall macro F1 — golden dataset",
                        color="Model", range_y=[0.5, 1.0])
fig_f1_overall.update_traces(textposition="outside")
fig_f1_overall.show()

# ── Per-category F1 grouped bar chart ─────────────────────────────────────────
per_cat_f1 = []
for name, df in results.items():
    for cat in sorted(df["produce"].unique()):
        sub = df[df["produce"] == cat]
        _, _, f1, _ = precision_recall_fscore_support(
            sub["true_label"], sub["pred"], average="binary", zero_division=0
        )
        per_cat_f1.append({"produce": cat, "F1": f1, "model": name})

per_cat_f1_df = pd.DataFrame(per_cat_f1)

fig_f1 = px.bar(per_cat_f1_df, x="produce", y="F1", color="model",
                barmode="group", title="Per-category F1 — golden dataset",
                range_y=[0, 1.0], text_auto=".2f")
fig_f1.update_layout(xaxis_tickangle=-45, height=500)
fig_f1.show()

# ── F1 heatmap ────────────────────────────────────────────────────────────────
pivot_f1 = per_cat_f1_df.pivot(index="model", columns="produce", values="F1")

fig_f1_heat = px.imshow(pivot_f1, text_auto=".2f", aspect="auto",
                        color_continuous_scale="RdYlGn", range_color=[0.4, 1.0],
                        title="F1 heatmap — model vs category")
fig_f1_heat.show()

## 2. Cross-Model Comparison
High-level comparison across all models: overall accuracy, per-category breakdown, and an accuracy heatmap. Key observation: the dirty dataset (largest volume) outperforms cleaner subsets, suggesting data volume dominates over data cleanliness for this task at current scale.

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import numpy as np

CLASS_NAMES = ["Healthy", "Rotten"]

for model_name, df in results.items():
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")

    y_true = df["true_label"].values
    y_pred = df["pred"].values
    conf   = df["confidence"].values

    # ── 1. Confusion matrix ───────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred)
    fig_cm = px.imshow(cm, text_auto=True,
                       x=CLASS_NAMES, y=CLASS_NAMES,
                       color_continuous_scale="Blues",
                       title=f"{model_name} — Confusion matrix",
                       labels=dict(x="Predicted", y="True"))
    fig_cm.update_layout(width=400, height=400)
    fig_cm.show()

    # ── 2. Precision / Recall / F1 per category ───────────────────
    categories = sorted(df["produce"].unique())
    rows = []
    for cat in categories:
        sub = df[df["produce"] == cat]
        p, r, f, _ = precision_recall_fscore_support(
            sub["true_label"], sub["pred"], average="binary", zero_division=0)
        rows.append({"Category": cat, "Precision": p, "Recall": r, "F1": f})
    # Macro average
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    rows.append({"Category": "MACRO AVG", "Precision": p_mac, "Recall": r_mac, "F1": f_mac})

    prf_df = pd.DataFrame(rows)
    fig_prf = go.Figure()
    for metric, color in [("Precision","#636EFA"), ("Recall","#EF553B"), ("F1","#00CC96")]:
        fig_prf.add_trace(go.Bar(
            name=metric, x=prf_df["Category"], y=prf_df[metric],
            marker_color=color, text=prf_df[metric].round(2), textposition="outside"
        ))
    fig_prf.update_layout(barmode="group", title=f"{model_name} — Precision / Recall / F1",
                          yaxis_range=[0, 1.1], xaxis_tickangle=-45, height=500)
    fig_prf.show()

    # ── 3. Confidence distribution ────────────────────────────────
    fig_conf = go.Figure()
    fig_conf.add_trace(go.Histogram(
        x=conf[df["correct"].values], name="Correct",
        marker_color="#00CC96", opacity=0.7, nbinsx=20))
    fig_conf.add_trace(go.Histogram(
        x=conf[~df["correct"].values], name="Incorrect",
        marker_color="#EF553B", opacity=0.7, nbinsx=20))
    fig_conf.update_layout(barmode="overlay",
                           title=f"{model_name} — Confidence distribution",
                           xaxis_title="Confidence", yaxis_title="Count")
    fig_conf.show()

    # ── 4. Error direction breakdown ──────────────────────────────
    errors = df[~df["correct"]].copy()
    errors["error_type"] = errors.apply(
        lambda r: "Healthy→Rotten" if r["true_label"] == 0 else "Rotten→Healthy", axis=1)
    total = df.groupby("produce").size()
    breakdown = errors.groupby(["produce","error_type"]).size().unstack(fill_value=0)
    breakdown_pct = (breakdown.div(total, axis=0) * 100).round(1)

    fig_err = px.bar(breakdown_pct.reset_index().melt(id_vars="produce"),
                     x="produce", y="value", color="error_type", barmode="group",
                     title=f"{model_name} — Error direction per category (%)",
                     labels={"value": "Error rate (%)", "produce": "Category"},
                     color_discrete_map={"Healthy→Rotten":"#EF553B","Rotten→Healthy":"#636EFA"})
    fig_err.update_layout(xaxis_tickangle=-45, height=450)
    fig_err.show()



  Dirty



  No identical



  No aug



  Progressive



  No dups


In [5]:
from scipy.optimize import minimize_scalar
import torch.nn.functional as F

# ── Temperature scaling calibration ───────────────────────────────────────────
# Calibrate the no-aug model (most overconfident).
# T is found by minimising NLL on the golden set — in practice you'd use a
# separate val set, but this demonstrates the technique on available data.

TARGET_MODEL = "No aug"

def evaluate_with_logits(model, transforms):
    """Same as evaluate() but also returns raw logits for calibration."""
    dataset = ProduceDataset(root_dir=GOLDEN_PATH, transform=transforms)
    loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            all_logits.append(model(imgs).cpu())
            all_labels.extend(labels.tolist())
    return torch.cat(all_logits), torch.tensor(all_labels)

# Reload the target model and collect logits
m, transforms = load_model(MODELS[TARGET_MODEL])
logits, labels = evaluate_with_logits(m, transforms)

# Find T that minimises NLL on the golden set
def nll(T):
    return F.cross_entropy(logits / T, labels).item()

result   = minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded")
best_T   = result.x
print(f"Optimal temperature T = {best_T:.3f}")

# ── Compare confidence distributions before and after ─────────────────────────
def conf_from_logits(logits, T=1.0):
    probs = torch.softmax(logits / T, dim=1)
    return probs.max(dim=1).values.numpy()

preds_orig = logits.argmax(dim=1).numpy()
correct    = (preds_orig == labels.numpy())

conf_before = conf_from_logits(logits, T=1.0)
conf_after  = conf_from_logits(logits, T=best_T)

fig_cal = make_subplots(rows=1, cols=2,
                        subplot_titles=["Before (T=1)", f"After (T={best_T:.2f})"])

for col, conf, title in [(1, conf_before, "Before"), (2, conf_after, "After")]:
    fig_cal.add_trace(go.Histogram(
        x=conf[correct],  name="Correct",   marker_color="#00CC96",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)
    fig_cal.add_trace(go.Histogram(
        x=conf[~correct], name="Incorrect", marker_color="#EF553B",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)

fig_cal.update_xaxes(range=[0.4, 1.0])
fig_cal.update_layout(barmode="overlay", title=f"{TARGET_MODEL} — confidence before vs after temperature scaling",
                      height=400)
fig_cal.show()

acc = correct.mean()
print(f"Accuracy before: {acc:.4f}")
print(f"Accuracy after:  {acc:.4f}  (unchanged — argmax is T-invariant)")

Optimal temperature T = 2.767


Accuracy before: 0.9119
Accuracy after:  0.9119  (unchanged — argmax is T-invariant)


## 3. Per-Model Detailed Analysis
For each model: confusion matrix, per-category Precision / Recall / F1, prediction confidence distribution, and error direction breakdown (Healthy→Rotten vs Rotten→Healthy).

The confidence distribution reveals **calibration** — a well-calibrated model should show high confidence on correct predictions and lower confidence on errors. Error direction shows whether the model leans toward false positives (calling rotten produce healthy) or false negatives.